In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [2]:
df = pd.read_csv("logistics_realworld_dataset.csv")
df.head()


,order_id,vendor,vehicle_type,origin_city,destination_city,shipment_date,planned_delivery,actual_delivery,distance_km,weather_condition,...,day_of_week,holiday_flag,pickup_delay_minutes,driver_rating,vehicle_age_years,order_weight_kg,num_packages,order_priority,road_type,is_delayed
0,ORD_100000,EcomExpress,Van,Delhi,Mumbai,27-06-2025 00:48,27-06-2025 16:48,27-06-2025 16:48,880,Clear,...,4,0,0,4.1,4,37.28,7,Standard,Highway,0
1,ORD_100001,EcomExpress,Tempo,Chennai,Kolkata,14-06-2025 01:48,15-06-2025 08:48,15-06-2025 08:48,1146,Fog,...,5,0,0,3.3,10,25.02,1,Standard,Highway,0
2,ORD_100002,BlueDart,Bike,Mumbai,Delhi,18-06-2025 10:48,19-06-2025 03:48,19-06-2025 05:48,778,Clear,...,2,0,5,3.7,10,19.38,8,Standard,Urban,0
3,ORD_100003,BlueDart,Van,Mumbai,Hyderabad,01-07-2025 14:48,02-07-2025 04:48,02-07-2025 16:48,731,Storm,...,1,0,10,2.8,2,38.39,2,Standard,Highway,1
4,ORD_100004,Shadowfax,Van,Mumbai,Delhi,03-07-2025 17:48,04-07-2025 09:48,04-07-2025 09:48,951,Storm,...,3,0,0,3.3,5,3.22,4,Standard,Highway,0


In [3]:
df.info()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_id              10000 non-null  object 
 1   vendor                10000 non-null  object 
 2   vehicle_type          10000 non-null  object 
 3   origin_city           10000 non-null  object 
 4   destination_city      10000 non-null  object 
 5   shipment_date         10000 non-null  object 
 6   planned_delivery      10000 non-null  object 
 7   actual_delivery       10000 non-null  object 
 8   distance_km           10000 non-null  int64  
 9   weather_condition     10000 non-null  object 
 10  traffic_condition     10000 non-null  object 
 11  vendor_delay_score    10000 non-null  float64
 12  hour_of_day           10000 non-null  int64  
 13  day_of_week           10000 non-null  int64  
 14  holiday_flag          10000 non-null  int64  
 15  pickup_delay_minutes

,distance_km,vendor_delay_score,hour_of_day,day_of_week,holiday_flag,pickup_delay_minutes,driver_rating,vehicle_age_years,order_weight_kg,num_packages,is_delayed
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.0,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,752.192100,0.323838,11.464400,3.025600,0.0,3.909500,3.749170,5.020200,25.307144,5.493200,0.248300
std,434.184481,0.157910,6.937618,1.938226,0.0,6.466645,0.720492,3.175059,14.354856,2.872626,0.432048
min,10.000000,0.050000,0.000000,0.000000,0.0,0.000000,2.500000,0.000000,0.500000,1.000000,0.000000
25%,372.000000,0.190000,5.000000,1.000000,0.0,0.000000,3.100000,2.000000,12.767500,3.000000,0.000000
50%,751.000000,0.320000,12.000000,3.000000,0.0,0.000000,3.700000,5.000000,25.145000,6.000000,0.000000
75%,1130.000000,0.460000,17.000000,5.000000,0.0,5.000000,4.400000,8.000000,37.722500,8.000000,0.000000
max,1500.000000,0.600000,23.000000,6.000000,0.0,30.000000,5.000000,10.000000,50.000000,10.000000,1.000000


In [5]:
df['is_delayed'].value_counts()


is_delayed
0    7517
1    2483
Name: count, dtype: int64

In [6]:
df.isnull().sum()


order_id                0
vendor                  0
vehicle_type            0
origin_city             0
destination_city        0
shipment_date           0
planned_delivery        0
actual_delivery         0
distance_km             0
weather_condition       0
traffic_condition       0
vendor_delay_score      0
hour_of_day             0
day_of_week             0
holiday_flag            0
pickup_delay_minutes    0
driver_rating           0
vehicle_age_years       0
order_weight_kg         0
num_packages            0
order_priority          0
road_type               0
is_delayed              0
dtype: int64

In [7]:
# Numeric columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categorical columns
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


These must not go into ML:
order_id → identifier
shipment_date, planned_delivery, actual_delivery → data leakage

“Identifiers and post-delivery timestamps were removed to prevent data leakage.”

In [ ]:
df.drop(columns=['order_id''shipment_date',
        'planned_delivery',
        'actual_delivery'], inplace=True)


In [12]:
from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include='object').columns

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])


In [13]:
X = df.drop(columns=['is_delayed'])
y = df['is_delayed']

print(X.shape, y.shape)


(10000, 18) (10000,)


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [16]:
lr = LogisticRegression(max_iter=3000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_pred))

Logistic Regression Accuracy: 0.7515


In [18]:
rf_balanced = RandomForestClassifier(
    n_estimators=200,
    max_depth=14,
    random_state=42,
    class_weight='balanced'
)

rf_balanced.fit(X_train, y_train)
rf_pred_bal = rf_balanced.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

print("Balanced RF Accuracy:", accuracy_score(y_test, rf_pred_bal))
print("\nClassification Report:\n", classification_report(y_test, rf_pred_bal))


Balanced RF Accuracy: 0.751

Classification Report:
               precision    recall  f1-score   support

           0       0.75      1.00      0.86      1503
           1       0.33      0.00      0.00       497

    accuracy                           0.75      2000
   macro avg       0.54      0.50      0.43      2000
weighted avg       0.65      0.75      0.65      2000



In [20]:
from sklearn.metrics import classification_report

# Predict probabilities
y_probs = rf_balanced.predict_proba(X_test)[:, 1]

# Lower threshold to catch delays
y_pred_threshold = (y_probs >= 0.30).astype(int)

print("Threshold = 0.30")
print(classification_report(y_test, y_pred_threshold))

Threshold = 0.30
              precision    recall  f1-score   support

           0       0.72      0.28      0.40      1503
           1       0.24      0.67      0.35       497

    accuracy                           0.38      2000
   macro avg       0.48      0.48      0.38      2000
weighted avg       0.60      0.38      0.39      2000



In [21]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(rf_balanced, f)

print("Model saved successfully")

Model saved successfully
